# 03 -- Backtesting, GARCH Volatility & Stress Testing

This notebook completes the VaR model validation pipeline by addressing three
critical questions that every risk manager must answer:

1. **Is the VaR model accurate?** -- Backtesting compares ex-ante VaR
   forecasts against ex-post realised returns to detect systematic
   under- or over-estimation of risk.  We apply three complementary
   statistical tests (Kupiec POF, Christoffersen conditional coverage,
   and the Basel traffic-light classification) to evaluate both the
   *frequency* and the *clustering* of violations.

2. **Does the model capture time-varying volatility?** -- The constant-
   volatility assumption underlying basic parametric VaR is well known to
   be unrealistic.  GARCH-family models (GARCH, EGARCH, GJR-GARCH)
   provide a more faithful representation of volatility clustering and
   leverage effects, which directly improves VaR forecasts during
   turbulent markets.

3. **What happens under extreme but plausible conditions?** -- Stress
   testing goes *beyond* VaR by evaluating portfolio losses during
   historical crises (COVID-19 crash, 2022 rate-hike cycle) and under
   hypothetical shocks (market crash, correlation spike, volatility
   doubling, flight to quality).  Reverse stress testing works backwards
   from a maximum tolerable loss to identify the scenarios that could
   cause it.

Together, backtesting and stress testing form the twin pillars of model
validation required by Basel III/IV and sound risk management practice.

In [ ]:
# ---------------------------------------------------------------------------
# Imports & configuration
# ---------------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from var_risk_engine.data import fetch_and_prepare
from var_risk_engine.var_historical import historical_var
from var_risk_engine.var_parametric import parametric_var
from var_risk_engine.backtesting import (
    rolling_backtest, kupiec_pof_test, christoffersen_test,
    basel_traffic_light, count_violations,
)
from var_risk_engine.stress_testing import (
    historical_stress_scenarios, hypothetical_stress,
    reverse_stress_test, stress_comparison_table,
)
from var_risk_engine.volatility import (
    fit_garch11, conditional_volatility, compare_garch_models,
)

# --- Professional financial styling (consistent with other notebooks) -------
sns.set_theme(style="whitegrid", palette="muted")
COLORS = {
    "primary":    "#1B3A5C",
    "secondary":  "#E8734A",
    "tertiary":   "#4CAF50",
    "quaternary": "#9C27B0",
    "bg":         "#FAFAFA",
}

plt.rcParams.update({
    "figure.facecolor": COLORS["bg"],
    "axes.facecolor":   COLORS["bg"],
    "font.size": 11,
})

# --- Fetch data & compute portfolio returns ---------------------------------
TICKERS = ["AAPL", "MSFT", "SPY", "TLT", "GLD"]
WEIGHTS = np.array([0.25, 0.25, 0.20, 0.15, 0.15])
CONFIDENCE = 0.95

prices, returns = fetch_and_prepare(TICKERS, start="2019-01-01")
port_returns = returns.values @ WEIGHTS
port_series  = pd.Series(port_returns, index=returns.index, name="portfolio")

print(f"Data range : {returns.index[0].date()} to {returns.index[-1].date()}")
print(f"Trading days: {len(returns)}")
print(f"Assets     : {TICKERS}")
print(f"Weights    : {WEIGHTS}")
print(f"Portfolio return mean={np.mean(port_returns):.5f}  std={np.std(port_returns):.5f}")

---
## Part A: Backtesting

We perform a rolling T+1 backtest using Historical VaR with a 250-day
estimation window at 95% confidence.  For each day beyond the initial
window the model forecasts one-day VaR from the trailing 250
observations, which is then compared to the realised return.

In [ ]:
# ---------------------------------------------------------------------------
# Rolling backtest -- Historical VaR (250-day window, 95% confidence)
# ---------------------------------------------------------------------------
WINDOW = 250

bt = rolling_backtest(
    returns=port_returns,
    var_func=historical_var,
    confidence=CONFIDENCE,
    window=WINDOW,
)

n_total  = len(bt["var_forecasts"])
n_viol   = bt["n_violations"]
expected = n_total * (1 - CONFIDENCE)

print(f"Rolling backtest ({WINDOW}-day window, {CONFIDENCE:.0%} confidence)")
print(f"  Forecasts         : {n_total}")
print(f"  Violations        : {n_viol}")
print(f"  Expected violations: {expected:.1f}")
print(f"  Violation rate     : {n_viol / n_total:.4f}  (target: {1 - CONFIDENCE:.4f})")

In [ ]:
# ---------------------------------------------------------------------------
# Plot: actual returns vs VaR band with violation markers
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 5))

dates = bt["dates_idx"]

# VaR band (negative because VaR is reported as a positive loss)
ax.fill_between(
    dates, -bt["var_forecasts"], 0,
    color=COLORS["primary"], alpha=0.12, label="VaR band (95%)",
)

# Actual returns
ax.plot(
    dates, bt["actual_returns"],
    color=COLORS["primary"], linewidth=0.7, alpha=0.8,
    label="Actual daily returns",
)

# Violations highlighted in red
viol_mask = bt["violations"]
ax.scatter(
    dates[viol_mask], bt["actual_returns"][viol_mask],
    color=COLORS["secondary"], s=30, zorder=5, edgecolors="darkred",
    linewidths=0.5, label=f"Violations ({n_viol})",
)

ax.set_xlabel("Trading Day Index", fontsize=11)
ax.set_ylabel("Daily Return", fontsize=11)
ax.set_title(
    f"Historical VaR Backtest -- {n_viol} violations / {n_total} days",
    fontsize=13, color=COLORS["primary"],
)
ax.legend(fontsize=9, loc="upper right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=1))
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Kupiec Proportion-of-Failures (POF) test
# ---------------------------------------------------------------------------
kupiec = kupiec_pof_test(
    bt["actual_returns"], bt["var_forecasts"], confidence=CONFIDENCE,
)

print("=" * 55)
print("  Kupiec POF Test (Proportion of Failures)")
print("=" * 55)
print(f"  H0: The observed violation rate equals the expected rate ({1 - CONFIDENCE:.0%}).")
print()
print(f"  Observations       : {kupiec['n_obs']}")
print(f"  Violations observed: {kupiec['n_violations']}")
print(f"  Expected violations: {kupiec['expected_violations']:.1f}")
print(f"  Actual viol. rate  : {kupiec['actual_rate']:.4f}")
print(f"  Expected rate      : {1 - CONFIDENCE:.4f}")
print(f"  LR test statistic  : {kupiec['test_statistic']:.4f}")
print(f"  p-value            : {kupiec['p_value']:.4f}")
print()
if kupiec["reject_h0"]:
    print("  >> REJECT H0 at 5% significance.")
    print("     The model systematically mis-estimates risk.")
else:
    print("  >> FAIL TO REJECT H0 at 5% significance.")
    print("     The violation frequency is consistent with the model.")

In [ ]:
# ---------------------------------------------------------------------------
# Christoffersen conditional coverage test
# ---------------------------------------------------------------------------
christ = christoffersen_test(
    bt["actual_returns"], bt["var_forecasts"], confidence=CONFIDENCE,
)

print("=" * 55)
print("  Christoffersen Conditional Coverage Test")
print("=" * 55)
print("  H0: Violations are correctly distributed AND independent.")
print()
print(f"  LR test statistic  : {christ['test_statistic']:.4f}")
print(f"  p-value            : {christ['p_value']:.4f}")
print(f"  Reject H0 (5%)     : {christ['reject_h0']}")
print()

# Transition matrix
trans_matrix = pd.DataFrame(
    [[christ["n00"], christ["n01"]],
     [christ["n10"], christ["n11"]]],
    index=["No Violation (t)", "Violation (t)"],
    columns=["No Violation (t+1)", "Violation (t+1)"],
)
print("  Transition Count Matrix:")
print(trans_matrix.to_string())
print()

# Transition probabilities
n0_total = christ["n00"] + christ["n01"]
n1_total = christ["n10"] + christ["n11"]
pi_01 = christ["n01"] / n0_total if n0_total > 0 else 0
pi_11 = christ["n11"] / n1_total if n1_total > 0 else 0
print(f"  P(violation at t+1 | no violation at t) = {pi_01:.4f}")
print(f"  P(violation at t+1 | violation at t)     = {pi_11:.4f}")
print()
if christ["n11"] > 0:
    print("  NOTE: n11 > 0 indicates violation clustering -- violations tend")
    print("  to be followed by further violations, suggesting the model does")
    print("  not adapt quickly enough to changing volatility regimes.")
else:
    print("  No violation clustering detected (n11 = 0).")

In [ ]:
# ---------------------------------------------------------------------------
# Basel traffic light classification
# ---------------------------------------------------------------------------
tl = basel_traffic_light(n_viol, n_total)

zone_colors = {"green": "#2E7D32", "yellow": "#F9A825", "red": "#C62828"}

print("=" * 55)
print("  Basel Traffic Light Classification")
print("=" * 55)
print(f"  Violations (250-day, 99% VaR basis): {tl['n_violations']}")
print(f"  Zone             : {tl['zone'].upper()}")
print(f"  Scaling factor   : {tl['scaling_factor']:.2f}x")
print(f"  Interpretation   : {tl['interpretation']}")
print()

# Visual indicator
fig, ax = plt.subplots(figsize=(6, 1.2))
for i, (zone, color) in enumerate(zone_colors.items()):
    alpha = 1.0 if zone == tl["zone"] else 0.2
    ax.barh(0, 1, left=i, color=color, alpha=alpha, edgecolor="white", height=0.6)
    ax.text(i + 0.5, 0, zone.upper(), ha="center", va="center",
            fontsize=10, fontweight="bold", color="white")
ax.set_xlim(0, 3)
ax.set_ylim(-0.5, 0.5)
ax.axis("off")
ax.set_title(
    f"Basel Zone: {tl['zone'].upper()}  "
    f"(scaling factor {tl['scaling_factor']:.2f}x)",
    fontsize=12, color=COLORS["primary"],
)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Cumulative violations vs expected (linear) with acceptable band
# ---------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(14, 4))

cum_violations = np.cumsum(bt["violations"].astype(int))
expected_cum   = np.arange(1, n_total + 1) * (1 - CONFIDENCE)

# Acceptable band: 0.5x to 1.5x expected
ax.fill_between(
    dates, expected_cum * 0.5, expected_cum * 1.5,
    alpha=0.12, color=COLORS["tertiary"],
    label="Acceptable band (0.5x - 1.5x expected)",
)

ax.plot(
    dates, cum_violations,
    color=COLORS["secondary"], linewidth=1.8,
    label=f"Cumulative violations (actual: {n_viol})",
)
ax.plot(
    dates, expected_cum,
    color=COLORS["tertiary"], linewidth=1.8, linestyle="--",
    label=f"Expected cumulative (linear, ~{expected:.0f})",
)

ax.set_xlabel("Trading Day Index", fontsize=11)
ax.set_ylabel("Cumulative Violations", fontsize=11)
ax.set_title(
    "Cumulative VaR Violations vs Expected (Linear)",
    fontsize=13, color=COLORS["primary"],
)
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout()
plt.show()

### Interpreting the Backtesting Results

We applied three complementary tests to evaluate the VaR model:

| Test | What it measures | Null hypothesis | Distribution |
|------|------------------|-----------------|-------------|
| **Kupiec POF** | Whether the *proportion* of violations matches the expected rate | Observed violation rate = 1 - confidence | Chi-squared(1) |
| **Christoffersen CC** | Whether violations are *independent* (no clustering) **and** have the correct unconditional rate | Violations are i.i.d. with correct probability | Chi-squared(2) |
| **Basel Traffic Light** | Regulatory classification into Green / Yellow / Red zones based on the absolute number of violations | N/A (rule-based) | N/A |

**Key takeaways:**

- The **Kupiec POF** test tells us whether the model gets the *average* violation
  rate right, but it cannot detect whether violations cluster in time.
- The **Christoffersen** test adds a check for *independence*: if violations
  tend to follow other violations (high $n_{11}$), the model is slow to adapt
  to regime changes -- a common problem with fixed-window Historical VaR
  during volatility spikes.
- The **Basel traffic light** is a simple rule-based classification that maps
  the number of violations to a capital scaling factor.  A yellow or red
  zone result directly increases the bank's required market-risk capital.

If both Kupiec and Christoffersen fail to reject $H_0$ and the Basel zone is
green, the model passes validation.  Any rejection warrants investigation
into whether the estimation window, confidence level, or VaR method should
be adjusted.

---
## Part B: GARCH Volatility

Historical VaR treats all past observations equally within the estimation
window.  GARCH models, by contrast, assign *time-varying* weights that
capture volatility clustering -- large moves tend to be followed by large
moves, and small by small.  This section fits GARCH(1,1) to the portfolio
return series and compares model fit statistics across assets.

In [ ]:
# ---------------------------------------------------------------------------
# Fit GARCH(1,1) on portfolio returns and extract conditional volatility
# ---------------------------------------------------------------------------
garch_result = fit_garch11(port_series, p=1, q=1, dist="studentst")
cond_vol = conditional_volatility(garch_result)

print("GARCH(1,1) -- Portfolio (Student-t distribution)")
print("-" * 50)
print(garch_result.summary().as_text())
print()
print(f"Unconditional vol (daily) : {np.std(port_returns):.5f}")
print(f"Latest conditional vol   : {cond_vol.iloc[-1]:.5f}")
print(f"Mean conditional vol     : {cond_vol.mean():.5f}")
print(f"Max  conditional vol     : {cond_vol.max():.5f}")

In [ ]:
# ---------------------------------------------------------------------------
# Dual-axis chart: returns on top, conditional vol on bottom
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(
    2, 1, figsize=(14, 8), sharex=True,
    gridspec_kw={"height_ratios": [1, 1]},
)

# --- Top panel: daily returns ---
ax = axes[0]
ax.plot(
    port_series.index, port_series.values,
    color=COLORS["primary"], linewidth=0.6, alpha=0.7,
    label="Daily portfolio returns",
)
cv_aligned = cond_vol.reindex(port_series.index)
ax.fill_between(
    port_series.index,
    -cv_aligned.values * 1.645,
     cv_aligned.values * 1.645,
    alpha=0.10, color=COLORS["secondary"],
    label=r"$\pm 1.645\,\sigma_t$ band (95%)",
)
ax.set_ylabel("Daily Return", fontsize=11)
ax.set_title(
    "Portfolio Returns vs GARCH(1,1) Conditional Volatility",
    fontsize=13, color=COLORS["primary"],
)
ax.legend(fontsize=9, loc="upper right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=1))

# --- Bottom panel: conditional volatility ---
ax = axes[1]
ax.plot(
    cond_vol.index, cond_vol.values,
    color=COLORS["secondary"], linewidth=1.0,
    label="Conditional volatility (GARCH)",
)
ax.axhline(
    np.std(port_returns),
    color=COLORS["tertiary"], linestyle="--", linewidth=1.5,
    label=f"Unconditional vol = {np.std(port_returns):.4f}",
)
ax.set_ylabel("Volatility (daily)", fontsize=11)
ax.set_xlabel("Date", fontsize=11)
ax.set_title(
    "Time-Varying Volatility -- GARCH vs Constant (Unconditional)",
    fontsize=11, color=COLORS["primary"],
)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Compare GARCH / EGARCH / GJR-GARCH for selected assets
# ---------------------------------------------------------------------------
comparison_assets = ["SPY", "AAPL", "TLT"]
returns_dict = {t: returns[t] for t in comparison_assets}

print("Fitting GARCH(1,1), EGARCH(1,1), GJR-GARCH(1,1,1) with Student-t ...")
print()

model_comparison = compare_garch_models(returns_dict)

display_cols = ["ticker", "model", "aic", "bic", "loglikelihood", "alpha", "beta", "gamma"]
print(model_comparison[display_cols].to_string(index=False, float_format="%.4f"))
print()

# Per-asset best model (lowest BIC)
print("Best model per asset (lowest BIC):")
print("-" * 40)
for ticker in comparison_assets:
    sub = model_comparison[model_comparison["ticker"] == ticker]
    best = sub.loc[sub["bic"].idxmin()]
    print(
        f"  {ticker:6s}  ->  {best['model']:12s}  "
        f"(BIC={best['bic']:.2f})"
    )

---
## Part C: Stress Testing

VaR captures risk under "normal" market conditions, but extreme events --
market crashes, liquidity crises, policy shocks -- can produce losses far
beyond what VaR predicts.  Stress testing evaluates portfolio resilience
under three complementary approaches:

1. **Historical scenarios** -- replay actual crisis periods from the data.
2. **Hypothetical scenarios** -- apply engineered shocks to test specific
   risk channels (directional moves, correlation breakdowns, vol spikes).
3. **Reverse stress testing** -- start from a maximum tolerable loss and
   work backwards to identify what could cause it.

In [ ]:
# ---------------------------------------------------------------------------
# Historical stress scenarios (COVID, 2022 rate hikes, worst 20-day)
# ---------------------------------------------------------------------------
hist_stress = historical_stress_scenarios(returns, WEIGHTS)

print("Historical Stress Scenarios")
print("=" * 65)
print(hist_stress.to_string(float_format="%.4f"))
print()
for scenario_name, row in hist_stress.iterrows():
    note = row.get("note", "")
    if note:
        print(f"  {scenario_name}: {note}")

In [ ]:
# ---------------------------------------------------------------------------
# Hypothetical stress scenarios
# ---------------------------------------------------------------------------
hypo_stress = hypothetical_stress(returns, WEIGHTS)

print("Hypothetical Stress Scenarios")
print("=" * 65)
print(hypo_stress.to_string(float_format="%.4f"))
print()
for scenario_name, row in hypo_stress.iterrows():
    print(f"  {scenario_name}:")
    print(
        f"    Portfolio return : {row['portfolio_return']:+.4f}  "
        f"({row['loss_bps']:.0f} bps)"
    )
    print(f"    Stressed VaR 95% : {row['var_95']:.4f}")
    print(f"    Stressed ES  95% : {row['es_95']:.4f}")
    print(f"    Note             : {row['note']}")
    print()

In [ ]:
# ---------------------------------------------------------------------------
# Reverse stress test (max_loss = 10%)
# ---------------------------------------------------------------------------
MAX_LOSS = 0.10

rst = reverse_stress_test(returns, WEIGHTS, max_loss=MAX_LOSS)

print("=" * 55)
print(f"  Reverse Stress Test  (max tolerable loss = {MAX_LOSS:.0%})")
print("=" * 55)
print(f"  Percentile in hist. dist. : {rst['percentile']:.1f}th")
print(f"  Uniform asset drop req.   : {rst['uniform_drop_required']:.2%}")
print(f"  Volatility multiplier     : {rst['vol_multiplier']:.2f}x")
print()
print("  Interpretation:")
print(f"  {rst['interpretation']}")

In [ ]:
# ---------------------------------------------------------------------------
# Master stress comparison table
# ---------------------------------------------------------------------------
master_table = stress_comparison_table(returns, WEIGHTS)

print("Stress Comparison Table (sorted worst-first)")
print("=" * 75)
print(master_table.to_string(index=False, float_format="%.4f"))

In [ ]:
# ---------------------------------------------------------------------------
# Stress scenario visualisation -- bar charts
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Historical stress -- cumulative return ---
ax = axes[0]
hist_clean = hist_stress.dropna(subset=["cumulative_return"])
if not hist_clean.empty:
    bar_colors = [
        COLORS["secondary"] if v < 0 else COLORS["tertiary"]
        for v in hist_clean["cumulative_return"]
    ]
    bars = ax.barh(
        range(len(hist_clean)),
        hist_clean["cumulative_return"],
        color=bar_colors, edgecolor="white",
    )
    ax.set_yticks(range(len(hist_clean)))
    ax.set_yticklabels(hist_clean.index, fontsize=9)
    ax.set_xlabel("Cumulative Return", fontsize=11)
    ax.set_title(
        "Historical Stress -- Cumulative Portfolio Return",
        fontsize=12, color=COLORS["primary"],
    )
    ax.axvline(0, color="black", linewidth=0.8)
    for bar, val in zip(bars, hist_clean["cumulative_return"]):
        x_pos = val - 0.005 if val < 0 else val + 0.005
        ha = "right" if val < 0 else "left"
        ax.text(
            x_pos, bar.get_y() + bar.get_height() / 2,
            f"{val:.1%}", va="center", ha=ha,
            fontsize=9, fontweight="bold",
        )

# --- Right: Hypothetical stress -- VaR & ES comparison ---
ax = axes[1]
hypo_clean = hypo_stress[["var_95", "es_95"]].dropna()
if not hypo_clean.empty:
    x_pos = np.arange(len(hypo_clean))
    width = 0.35
    ax.bar(
        x_pos - width / 2, hypo_clean["var_95"], width,
        label="VaR (95%)", color=COLORS["secondary"], edgecolor="white",
    )
    ax.bar(
        x_pos + width / 2, hypo_clean["es_95"], width,
        label="ES (95%)", color=COLORS["quaternary"], edgecolor="white",
    )
    ax.set_xticks(x_pos)
    ax.set_xticklabels(
        hypo_clean.index, fontsize=8, rotation=15, ha="right",
    )
    ax.set_ylabel("Risk Measure", fontsize=11)
    ax.set_title(
        "Hypothetical Stress -- Stressed VaR & ES",
        fontsize=12, color=COLORS["primary"],
    )
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Key Findings: Model Adequacy & Stress Test Results

**Backtesting**

- The rolling Historical VaR backtest provides a direct assessment of
  model calibration.  The Kupiec POF test evaluates whether the
  *frequency* of violations is consistent with the confidence level,
  while the Christoffersen test additionally checks for violation
  *independence*.  If violations cluster (high $n_{11}$), the model
  is slow to adapt -- a known limitation of fixed-window Historical VaR
  during regime shifts.
- The Basel traffic-light result translates directly into capital
  implications: yellow and red zones require a multiplicative scaling
  factor on the VaR-based capital charge.

**GARCH Volatility**

- GARCH(1,1) captures volatility clustering that is invisible to the
  unconditional (constant) volatility estimate.  The conditional
  volatility spikes during crisis periods (COVID-19, 2022 rate hikes)
  confirm that the model adapts to regime changes.
- EGARCH and GJR-GARCH add asymmetric (leverage) effects.  For equity-
  heavy assets (SPY, AAPL) these models typically show a negative gamma,
  meaning that negative returns increase volatility more than positive
  returns of the same magnitude.  For bonds (TLT) the asymmetry is
  usually weaker.
- Model selection via BIC balances goodness-of-fit against complexity.

**Stress Testing**

- Historical stress scenarios reveal the portfolio's actual behaviour
  during documented crises.  The COVID-19 crash typically produces the
  largest cumulative loss and worst single-day return.
- Hypothetical scenarios test specific risk channels: the Market Crash
  scenario shows the impact of a broad equity sell-off; the Correlation
  Spike scenario demonstrates that diversification benefits evaporate
  when correlations approach 1; Volatility Doubling shows how risk
  measures scale; and Flight to Quality illustrates that bonds and gold
  can partially offset equity losses.
- The reverse stress test identifies the conditions under which the
  portfolio hits a pre-specified loss threshold, providing actionable
  insight for risk-limit setting and contingency planning.

**Overall assessment**: a VaR model that passes backtesting tests,
incorporates GARCH-based volatility dynamics, and survives plausible
stress scenarios provides a robust foundation for market-risk measurement
and regulatory capital calculation.